git add nanoGPT-master/
git commit -m "更新说明"
git push origin main

In [1]:
with open('input.txt','r',encoding = 'utf-8') as f:
    text = f.read()

print('length of dataset in characters: ',len(text))

length of dataset in characters:  1115394


![](images/2026-10-11-10-55-27.png)
#encoding='utf-8' 是一个用来“翻译文本数据”的字符编码规则
![](images/2026-10-11-11-38-40.png)
![](images/2026-10-11-11-41-04.png)
![](images/2026-10-11-11-41-42.png)
![](images/2026-10-11-11-41-56.png)
![](images/2026-10-11-11-42-27.png)
![](images/2026-10-11-11-42-48.png)

In [2]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [ ]:
#字符编码器与解码器  tokenizer分词器
#把人类看懂的文本（字符 char），转换成神经网络看得懂的数字（整数 integer），以及反向转换回去
#专门构建一个针对该文本的小密码字典，而不是直接用 UTF-8
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l:''.join([itos[i] for i in l])

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


![](images/2026-10-11-15-36-37.png)
![](images/2026-10-11-14-52-46.png)
![](images/2026-10-11-14-54-04.png)
![](images/2026-10-11-14-54-31.png)
![](images/2026-10-11-14-54-42.png)
![](images/2026-10-11-14-54-59.png)
![](images/2026-10-11-15-36-56.png)

![](images/2026-10-11-15-21-28.png)
![](images/2026-10-11-15-27-13.png)
![](images/2026-10-11-15-27-41.png)
![](images/2026-10-11-15-28-07.png)
![](images/2026-10-11-15-29-33.png)
![](images/2026-10-11-15-32-23.png)
![](images/2026-10-11-15-32-40.png)
![](images/2026-10-11-15-33-25.png)

In [4]:
import torch
data = torch.tensor(encode(text),dtype = torch.long)
print(data.shape,data.dtype)
print(data[:100])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


![](images/2026-10-11-15-51-16.png)
![](images/2026-10-11-15-51-39.png)
#PyTorch 创建张量时，默认数据类型的确常常是 torch.float32

In [6]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [8]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [12]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target:{target}")

when input is tensor([18]) the target:47
when input is tensor([18, 47]) the target:56
when input is tensor([18, 47, 56]) the target:57
when input is tensor([18, 47, 56, 57]) the target:58
when input is tensor([18, 47, 56, 57, 58]) the target:1
when input is tensor([18, 47, 56, 57, 58,  1]) the target:15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target:47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target:58


![](images/2026-10-11-17-39-17.png)
![](images/2026-10-11-17-43-45.png)
![](images/2026-10-11-21-22-31.png)
![](images/2026-10-11-21-22-47.png)
![](images/2026-10-11-21-23-21.png)

In [14]:
torch.manual_seed(1337)
batch_size = 4  # how many independent sequences will we process in parallel?
block_size = 8  # what is the maximum context length for predictions?

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data)-block_size,(batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+1+block_size] for i in ix])
    return x,y

xb,yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b,:t+1]
        target = yb[b,t]
        print(f"when input is{context.tolist()} the target: {target}")


inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----
when input is[24] the target: 43
when input is[24, 43] the target: 58
when input is[24, 43, 58] the target: 5
when input is[24, 43, 58, 5] the target: 57
when input is[24, 43, 58, 5, 57] the target: 1
when input is[24, 43, 58, 5, 57, 1] the target: 46
when input is[24, 43, 58, 5, 57, 1, 46] the target: 43
when input is[24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is[44] the target: 53
when input is[44, 53] the target: 56
when input is[44, 53, 56] the target: 1
when input is[44, 53, 56, 1] the target: 58
when input is[44, 53, 56, 1, 58] the target: 46
when input is[44, 53, 56, 1, 58, 4

![](images/2026-10-11-21-53-54.png)
![](images/2026-10-11-22-16-29.png)
![](images/2026-10-11-22-05-20.png)
![](images/2026-10-11-22-05-37.png)
![](images/2026-10-11-22-06-21.png)
![](images/2026-10-11-22-06-56.png)
![](images/2026-10-11-22-07-11.png)
![](images/2026-10-11-22-08-28.png)
![](images/2026-10-11-22-08-49.png)

![](images/2026-10-11-22-22-47.png)
![](images/2026-10-11-22-37-41.png)
![](images/2026-10-11-22-53-19.png)
![](images/2026-10-12-10-02-15.png)
![](images/2026-10-12-10-02-25.png)

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self,vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size,vocab_size)

    def forward(self,idx,targets = None):
        #idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx)  #(B,T,C) Batch_size Time/Block_size  Channel/Vocab_size

        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T) #targets = targets.view(-1)也可以
            loss = F.cross_entropy(logits,targets)
            
        return logits,loss

    def generate(self,idx,max_new_tokens):
        #idx is (B,T) array of indices in the current context
        for _ in range(max_new_tokens):
            #get the predictions
            logits,loss = self(idx)
            #focus only on the last time step
            logits = logits[:,-1,:] #becomes (B,C)
            #apply softmax to get probabilities
            probs = F.softmax(logits,dim=-1) #(B,C)
            #sample from the distribution
            idx_next = torch.multinomial(probs,num_samples=1) #(B,1)
            #append sampled index to the running sequence
            idx = torch.cat((idx,idx_next),dim = 1) #(B,T+1)
        return idx


m = BigramLanguageModel(vocab_size)
logits,loss = m(xb,yb)
print(logits.shape)
print(loss)

idx = torch.zeros((1,1),dtype=torch.long)
print(decode(m.generate(idx,max_new_tokens=100)[0].tolist()))

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

Sr?qP-QWktXoL&jLDJgOLVz'RIoDqHdhsV&vLLxatjscMpwLERSPyao.qfzs$Ys$zF-w,;eEkzxjgCKFChs!iWW.ObzDnxA Ms$3


![](images/2026-10-12-10-36-22.png)
![](images/2026-10-12-10-36-49.png)
![](images/2026-10-12-10-38-52.png)
![](images/2026-10-12-14-29-35.png)
![](images/2026-10-12-14-35-08.png)
![](images/2026-10-12-14-30-28.png)
![](images/2026-10-12-14-53-21.png)
![](images/2026-10-12-14-53-39.png)
![](images/2026-10-12-14-54-02.png)
![](images/2026-10-12-14-54-34.png)
![](images/2026-10-12-14-55-14.png)
![](images/2026-10-12-14-56-54.png)
![](images/2026-10-12-14-57-17.png)
![](images/2026-10-12-14-59-15.png)
![](images/2026-10-12-14-59-33.png)

![](images/2026-10-12-15-27-09.png)
![](images/2026-10-12-15-39-14.png)
![](images/2026-10-13-15-47-37.png)
![](images/2026-10-13-15-48-31.png)
![](images/2026-10-13-15-48-50.png)
![](images/2026-10-13-15-50-32.png)
![](images/2026-10-13-15-50-54.png)
![](images/2026-10-12-15-52-13.png)
![](images/2026-10-12-15-53-31.png)
![](images/2026-10-12-15-55-09.png)
![](images/2026-10-12-16-46-42.png)
![](images/2026-10-12-16-46-59.png)
![](images/2026-10-12-16-48-05.png)

![](images/2026-10-13-09-50-19.png)
![](images/2026-10-13-09-25-39.png)
![](images/2026-10-13-09-25-56.png)
![](images/2026-10-13-09-44-20.png)
![](images/2026-10-13-09-44-34.png)
![](images/2026-10-13-09-44-48.png)
![](images/2026-10-13-09-46-11.png)
![](images/2026-10-13-09-46-27.png)
![](images/2026-10-13-09-47-25.png)

![](images/2026-10-13-15-07-18.png)
![](images/2026-10-13-11-05-29.png)
![](images/2026-10-13-14-42-38.png)
![](images/2026-10-13-11-05-58.png)
![](images/2026-10-13-11-06-41.png)
![](images/2026-10-13-11-07-17.png)
![](images/2026-10-13-11-07-33.png)
![](images/2026-10-13-14-44-56.png)

![](images/2026-10-13-15-08-18.png)
![](images/2026-10-13-15-23-04.png)
#代表类别的维度（Category / Class Dimension，常记为 C），就是用来存放所有“候选选项得分”的那个维度！
![](images/2026-10-13-16-19-07.png)
![](images/2026-10-13-16-19-44.png)
![](images/2026-10-13-16-20-14.png)
![](images/2026-10-13-16-24-13.png)
![](images/2026-10-13-22-00-09.png)
![](images/2026-10-13-22-00-19.png)

![](images/2026-10-13-16-43-56.png)
![](images/2026-10-13-16-44-12.png)
![](images/2026-10-13-16-44-23.png)
![](images/2026-10-13-16-47-29.png)

In [22]:
#create a PyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(),lr = 1e-3)

![](images/2026-10-13-22-32-48.png)
![](images/2026-10-13-22-43-45.png)
![](images/2026-10-13-22-43-55.png)
权重衰减（Weight Decay）本质上就是机器学习中的“L2 正则化（L2 Regularization）
![](images/2026-10-14-09-34-58.png)

In [28]:
batch_size = 32
for steps in range(10000):
    xb,yb = get_batch('train')

    logits,loss = m(xb,yb)  # 前向传播算损失
    optimizer.zero_grad(set_to_none = True) # 优化器将旧梯度清零
    loss.backward()   # 损失函数计算新梯度
    optimizer.step()   # 优化器根据新梯度更新参数

print(loss.item()) 

2.4581871032714844


![](images/2026-10-14-09-52-34.png)
![](images/2026-10-14-10-23-09.png)
![](images/2026-10-14-10-23-22.png)
![](images/2026-10-14-09-52-51.png)
![](images/2026-10-14-09-58-27.png)

In [30]:
print(decode(m.generate(idx = torch.zeros((1,1),dtype = torch.long),max_new_tokens = 500)[0].tolist()))


NCUSothe me h t haudst ba tthe r:
Myos ct

ANG HENGAmicavO:
ORD level athathyore Itoth thausthe tengsord
Bukeng cuthe, lth fore.
Bup immast k? t
II stenold; shiveilber hyachisayouiceambershe' I toul:
There?

WAd hach y ariontise r!
Werd g I me.
Whtt as RIBE s?
Soughe to drsthowhewarul pr'deageexPAKI m; w h axRorint fe s ckede d ns s;d,
EThaghat ye wies;As
Ahotoou
CENGowont De bry n sesealan.
The, bousir, stere
CKINGour! t, yo dal,
Noter r n.

Wn, wheve miang CUENGe rt QUENGHepix sdamy t, obout m
